# Proyecto 3: Mejorando algoritmos con CUDA
**CC3069 Computación Paralela y Distribuida — Universidad del Valle de Guatemala, semestre II-2026**

**Equipo:** Daniel Oswaldo Juárez Herrera · Humberto Alexander de la Cruz · Nicolle Alexandra Gordillo

Este notebook acelera el **Laboratorio 3 de Data Science (reconocimiento del alfabeto ASL con CNN)** separando el flujo en dos fases con hardware distinto:

| Fase | Qué hace | Dónde corre | Cómo |
|---|---|---|---|
| **1. Preparación de datos** | leer, decodificar y redimensionar 23,200 JPEG | **CPU, varios núcleos** | `ProcessPoolExecutor` + `SharedMemory` (`src/preparacion.py`) |
| **2. Entrenamiento e inferencia** | convoluciones, productos de matrices, retropropagación, Adam | **GPU (CUDA)** | PyTorch + cuDNN (`src/modelos.py`, `src/entrenamiento.py`) y kernels propios en Numba CUDA (`src/kernels_cuda.py`) |

La línea base es la solución original: Keras/TensorFlow en CPU (`src/modelos_keras.py`, copia fiel de `lab3_original/Lab3_Final.ipynb`).

**Cómo está organizado:** las secciones 1 a 6 siguen los pasos del enunciado. Las mediciones repetidas (varias corridas por configuración) las hace `python -m src.correr_experimentos` y quedan en `resultados/`. Este notebook las analiza y además ejecuta en vivo cada fase para dejar evidencia.

In [1]:
# Configuración: rutas, dependencias y hardware disponible
import os, sys, time, json, inspect, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

RAIZ = os.path.abspath('.')
sys.path.insert(0, RAIZ)
os.environ.setdefault('ASL_DATASET_PATH', os.path.join(RAIZ, 'asl_alphabet_train'))

from src import preparacion as prep
from src import analisis as an
from src.monitor import MonitorRecursos
from src.benchmark import info_sistema

an.estilo()
pd.set_option('display.precision', 3)
print(json.dumps(info_sistema(), indent=1, ensure_ascii=False))
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

{
 "python": "3.13.7",
 "so": "Windows-11-10.0.26200-SP0",
 "cpu": "Intel64 Family 6 Model 140 Stepping 2, GenuineIntel",
 "nucleos_fisicos": 4,
 "nucleos_logicos": 8,
 "ram_gb": 16.9,
 "torch": "2.11.0+cu128",
 "cuda_torch": "12.8",
 "cudnn": 91900,
 "gpu": "NVIDIA GeForce MX450",
 "hilos_torch_cpu": 4
}
Sat Nov 14 08:07:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.60                 KMD Version: 610.60        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeF

## 1. Análisis de la solución secuencial existente

El Lab 3 es un notebook de 57 celdas. Sus partes principales son:

| Parte | Celdas del Lab 3 | Función |
|---|---|---|
| Carga del dataset | 1 | Localiza `asl_alphabet_train/` y lista las 29 clases. |
| EDA (Ejercicios 1–2) | 2–7 | Muestra ejemplos, brillo promedio por clase y letras parecidas. |
| Preprocesamiento (Ejercicio 3) | 8–9 | Submuestrea 800 imágenes por clase, las redimensiona a 64×64 y normaliza a [0,1]. |
| Partición | 11 | Divide 70/15/15 de forma estratificada con semilla 42. |
| Modelos CNN (Ejercicio 4) | 12–16 | CNN Base y CNN Profunda en Keras, entrenadas con Adam y EarlyStopping. |
| Red densa (Ejercicio 5) | 17–18 | MLP de referencia (no llegó a aprender: 3.45%). |
| Evaluación | 19–25 | Exactitud en prueba, curvas, matriz de confusión y reporte por clase. |
| Algoritmos clásicos (Ejercicio 6) | 26–33 | Escala de grises + PCA; Random Forest, SVM y KNN con scikit-learn. |
| Aumento de datos (Ejercicio 7) | 34–45 | Capas de aumento dentro del modelo; reentrenamiento de ambas CNN y ablación con espejo. |
| Fotos propias (Ejercicio 8) | 46–52 | Prueba fuera de dominio con 15 fotos del equipo. |
| Sesgo (Ejercicio 9) | 53–55 | Análisis de vecinos más cercanos y discusión. |

El trabajo pesado está en dos lugares: **el preprocesamiento** (decodificar miles de JPEG) y **el entrenamiento de las CNN** (Ejercicios 4 y 7), que en la máquina original tomó 52 de los ~60 minutos del notebook.

### 1.1 Operaciones del programa, de la imagen al resultado

| No. | Operación identificada | Descripción de su funcionamiento |
|---|---|---|
| 1 | Listado de archivos | `os.listdir` por cada una de las 29 carpetas (3,000 archivos cada una). |
| 2 | Submuestreo | `random.sample` elige 800 archivos por clase: 23,200 imágenes. |
| 3 | Lectura del archivo | Lee los bytes del JPEG del disco (~15 KB cada uno). |
| 4 | Decodificación JPEG | Pillow descomprime cada imagen a 200×200×3 píxeles (Huffman + IDCT por bloques de 8×8). |
| 5 | Redimensión | `resize((64, 64))` con filtro bicúbico: cada píxel de salida combina una vecindad del original. |
| 6 | Normalización | Convierte a `float32` y divide entre 255; escribe la imagen en el tensor `X` (23,200×64×64×3, 1.14 GB). |
| 7 | Partición estratificada | `train_test_split` sobre índices: 16,240 / 3,480 / 3,480. |
| 8 | Barajado y loteo | En cada época se barajan los índices y se toman lotes de 64 imágenes (254 lotes por época). |
| 9 | Convolución 3×3 + ReLU | Cada valor de salida es una suma de 9·C_in productos más un sesgo. Es la operación dominante (ver 1.2). |
| 10 | Max-pooling 2×2 | Reduce a la mitad el alto y el ancho tomando el máximo de cada ventana. |
| 11 | Capas densas | Productos de matrices: 8192→128 (Base) o 128→256 (Profunda), y luego →29. |
| 12 | BatchNorm / Dropout | (Profunda) normaliza por canal; apaga activaciones al azar durante el entrenamiento. |
| 13 | Softmax + entropía cruzada | Convierte logits en probabilidades y calcula la pérdida del lote. |
| 14 | Retropropagación | Gradientes de todas las capas en orden inverso; cuesta ~2 veces la propagación hacia adelante. |
| 15 | Actualización Adam | Actualiza cada parámetro con sus momentos (1.1 millones de parámetros en la CNN Base). |
| 16 | Validación por época | Propagación hacia adelante sobre 3,480 imágenes al final de cada época. |
| 17 | Inferencia y métricas | `predict` sobre prueba, `argmax`, exactitud, matriz de confusión. |

In [2]:
# 1.2 Cuántas veces se repite cada operación (conteos exactos para esta configuración)
def macs_conv(lado, c_in, c_out):
    return lado * lado * c_out * 9 * c_in

capas = {
    'CNN Base': [('Conv 3→32 @64×64', macs_conv(64, 3, 32)), ('Conv 32→64 @32×32', macs_conv(32, 32, 64)),
                 ('Conv 64→128 @16×16', macs_conv(16, 64, 128)), ('Densa 8192→128', 8192 * 128),
                 ('Densa 128→29', 128 * 29)],
    'CNN Profunda': [('Conv 3→32', macs_conv(64, 3, 32)), ('Conv 32→32', macs_conv(64, 32, 32)),
                     ('Conv 32→64', macs_conv(32, 32, 64)), ('Conv 64→64', macs_conv(32, 64, 64)),
                     ('Conv 64→128', macs_conv(16, 64, 128)), ('Conv 128→128', macs_conv(16, 128, 128)),
                     ('Densa 128→256', 128 * 256), ('Densa 256→29', 256 * 29)],
}
N_TRAIN, LOTE = 16_240, 64
for modelo, lista in capas.items():
    total = sum(m for _, m in lista)
    tabla_capas = pd.DataFrame([(n, m / 1e6, m / total) for n, m in lista],
                               columns=['Capa', 'MACs por imagen (millones)', '% del total'])
    print(f"{modelo}: {total / 1e6:.1f} M multiplicaciones-suma por imagen hacia adelante; "
          f"~{3 * 2 * total * N_TRAIN / 1e12:.1f} TFLOP por época (adelante + atrás ≈ 3×)")
    display(tabla_capas.style.format({'MACs por imagen (millones)': '{:.2f}', '% del total': '{:.1%}'}))

CNN Base: 42.3 M multiplicaciones-suma por imagen hacia adelante; ~4.1 TFLOP por época (adelante + atrás ≈ 3×)


,Capa,MACs por imagen (millones),% del total
0,Conv 3→32 @64×64,3.54,8.4%
1,Conv 32→64 @32×32,18.87,44.6%
2,Conv 64→128 @16×16,18.87,44.6%
3,Densa 8192→128,1.05,2.5%
4,Densa 128→29,0.00,0.0%


CNN Profunda: 154.6 M multiplicaciones-suma por imagen hacia adelante; ~15.1 TFLOP por época (adelante + atrás ≈ 3×)


,Capa,MACs por imagen (millones),% del total
0,Conv 3→32,3.54,2.3%
1,Conv 32→32,37.75,24.4%
2,Conv 32→64,18.87,12.2%
3,Conv 64→64,37.75,24.4%
4,Conv 64→128,18.87,12.2%
5,Conv 128→128,37.75,24.4%
6,Densa 128→256,0.03,0.0%
7,Densa 256→29,0.01,0.0%


| Operación repetitiva | Datos que procesa | Cantidad de repeticiones |
|---|---|---|
| Lectura + decodificación + redimensión | un JPEG de 200×200×3 | **23,200** (una por imagen seleccionada) |
| Normalización (÷255) | 12,288 valores por imagen | 23,200 imágenes = **285 millones** de divisiones |
| Paso de entrenamiento (adelante + atrás + Adam) | un lote de 64×64×64×3 | **254 por época** × 20–30 épocas × 4 modelos |
| Multiplicación-suma de la convolución | ventana 3×3×C_in × filtro | **42.3 M por imagen** (Base) / **154.6 M** (Profunda), ×16,240 imágenes ×3 (adelante y atrás) por época |
| Max-pooling | ventanas 2×2 | 3 capas × 64 imágenes × (32²·32 + 16²·64 + 8²·128) salidas por lote |
| Actualización de Adam | 1,145,693 parámetros (Base) / 327,933 (Profunda) | una vez por lote: **254 × parámetros** por época |
| Validación | 3,480 imágenes de validación | una vez por época |

**Lectura de las tablas:** el 89% del cómputo de la CNN Base está en dos convoluciones y el 98% del de la Profunda en sus seis convoluciones. Son millones de multiplicaciones-suma **independientes entre sí** (cada valor de salida se calcula sin esperar a los demás), que es exactamente el tipo de trabajo para el que existe una GPU. La preparación, en cambio, son 23,200 tareas independientes y **pesadas en lógica** (descompresión JPEG, ramas, acceso a disco), que encajan mejor en núcleos de CPU.